# Feature Visualization, Point Cloud & KNN Laplacian Notebook

This notebook provides:
1. **Point Cloud Feature Visualization**: Render features directly on 3D point clouds (without mesh faces).
2. **KNN Graph Laplacian Smoothing (L = D - W)**: Construct a KNN graph on point cloud coordinates, compute L=D-W, and smooth features.
3. **High Activation Masking**: Highlight top activations in Red while masking lower activations to White.
4. **1-to-1 Point Cloud Correspondence**: Nearest-neighbor correspondence matching between point clouds.

In [1]:
import os
# Set PyOpenGL platform for headless rendering
os.environ["PYOPENGL_PLATFORM"] = "egl"

In [2]:
import sys
sys.path.insert(0, "/data/home/user/Lubesh_22CS30065/btp2")

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
import meshplot as mp
from scipy.spatial import cKDTree
from pyFM.mesh import TriMesh

from core.preprocessing import process_geometry, normalize_pc
from models.asmae import ASMAE
from utils.dino_utils import get_vertex_dino_features
from utils.mesh import load_off

print("Imports successful. PyOpenGL platform:", os.environ.get("PYOPENGL_PLATFORM"))

Imports successful. PyOpenGL platform: egl


In [3]:
# Load ASMAE Trained Checkpoint
config_path     = "./config/SHREC/train_st_te_config.yaml"
checkpoint_path = "./checkpoints/st_te_model_SHREC.pth"

with open(config_path) as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt   = torch.load(checkpoint_path, map_location=device, weights_only=True)
feature_dim = ckpt["feature_dim"]

cfg = config.get("student_model", config.get("model", {}))
model = ASMAE(
    feature_dim=feature_dim,
    embed_dim=cfg["embed_dim"], depth=cfg["depth"], num_heads=cfg["num_heads"],
    decoder_embed_dim=cfg["decoder_embed_dim"], decoder_depth=cfg["decoder_depth"],
    decoder_num_heads=cfg["decoder_num_heads"], mlk_ratio=cfg["mlk_ratio"],
    num_mask_queries=cfg.get("num_mask_queries", 5000),
    encoder_k=cfg.get("encoder_k", 20), aamg_k=cfg.get("aamg_k", 10),
    aamg_emb_dim=cfg.get("aamg_emb_dim", 64), pos_embed_dim=cfg.get("pos_embed_dim", 64),
    temperature=cfg.get("temperature", 1.0)
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"ASMAE loaded. Feature dim: {feature_dim}")

ASMAE loaded. Feature dim: 53


In [4]:
# Load Shapes and Extract Features
off_dir = "./input/SHREC/off/"
obj_dir = "./input/SHREC/obj/k_10/"
k, t, nv = config["geometry"]["k"], config["geometry"]["t"], config["geometry"].get("neigvecs", 300)
shape_names = ["9", "31"]

shapes_info = []
for name in shape_names:
    print(f"Processing {name}...")
    mesh = TriMesh(off_dir + name + ".off", center=True, area_normalize=True)
    V, El, feat, _ = process_geometry(obj_dir + name + ".obj", k, t, nv, output_dir=config["output_dir"])
    V_norm = normalize_pc(V.copy())
    f_t = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(device)
    p_t = torch.tensor(V_norm, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        asmae_feat = model.extract_features(f_t, p_t).squeeze(0).cpu().numpy()

    V_raw, _, ITris = load_off(off_dir + name + ".off")
    dino_feat, vis, _ = get_vertex_dino_features(V_raw, ITris, device=device)
    
    asmae_feat_norm = asmae_feat / (np.linalg.norm(asmae_feat, axis=1, keepdims=True) + 1e-8)
    dino_feat_norm  = dino_feat / (np.linalg.norm(dino_feat,  axis=1, keepdims=True) + 1e-8)
    combined = np.concatenate([asmae_feat_norm, dino_feat_norm], axis=1)

    shapes_info.append({
        "name": name,
        "pos": V,
        "mesh": mesh,
        "asmae_feats": asmae_feat_norm,
        "dino_feats": dino_feat_norm,
        "features": combined
    })
    print(f"  Loaded {name} ({len(V)} points). Combined features: {combined.shape}")

Processing 9...
[./input/SHREC/obj/k_10/9.obj] Loading...
[./input/SHREC/obj/k_10/9.obj] Running FPS (k=50)...
[./input/SHREC/obj/k_10/9.obj] Computing HKS (t=100)...
[./input/SHREC/obj/k_10/9.obj] Feature dim after append: 53 (50 HKS + 3 XYZ)
[DINOv2] Loading model: dinov2_vitb14_reg on cuda...


Using cache found in /data/home/user/.cache/torch/hub/facebookresearch_dinov2_main
/data/home/user/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/data/home/user/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/data/home/user/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


[DINOv2] Model loaded.
  Loaded 9 (5202 points). Combined features: (5202, 896)
Processing 31...
[./input/SHREC/obj/k_10/31.obj] Loading...
[./input/SHREC/obj/k_10/31.obj] Running FPS (k=50)...
[./input/SHREC/obj/k_10/31.obj] Computing HKS (t=100)...
[./input/SHREC/obj/k_10/31.obj] Feature dim after append: 53 (50 HKS + 3 XYZ)
  Loaded 31 (5201 points). Combined features: (5201, 896)


In [5]:
# KNN Graph Laplacian Function: L = D - W for Point Clouds
def compute_knn_laplacian_smoothing(features, pos, k=15, alpha=0.3):
    """
    Constructs a K-Nearest Neighbor (KNN) Graph on point cloud coordinates ,
    computes the Graph Laplacian L = D - W, and applies Laplacian feature smoothing.
    """
    N = pos.shape[0]
    tree = cKDTree(pos)
    dists, idxs = tree.query(pos, k=k)
    
    sigma = np.mean(dists[:, 1:]) + 1e-8
    W = np.zeros((N, N), dtype=np.float64)
    for i in range(N):
        for j_idx, d in zip(idxs[i, 1:], dists[i, 1:]):
            w = np.exp(-(d**2) / (2 * sigma**2))
            W[i, j_idx] = w
            W[j_idx, i] = w
            
    deg = W.sum(axis=1)
    deg_inv_sqrt = np.power(deg, -0.5)
    deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.
    D_inv = np.diag(deg_inv_sqrt)
    L_norm = np.eye(N) - D_inv @ W @ D_inv
    
    # Smooth features: F_smooth = F - alpha * (L @ F)
    smoothed = features - alpha * (L_norm @ features)
    return smoothed

print("KNN Graph Laplacian (L = D - W) function ready.")

KNN Graph Laplacian (L = D - W) function ready.


In [6]:
# Feature Visualization (Supports both 3D Mesh and Point Cloud)
def visualize_feature_dim(shapes_info, dim, top_percentile=80, apply_laplacian=True, as_mesh=True):
    """
    Displays feature dimension `dim` rendered on either a 3D Mesh (with faces) 
    or a 3D Point Cloud.
    
    Un-activated vertices/points are Light Gray [0.85, 0.85, 0.85].
    Optionally applies KNN Graph Laplacian smoothing (L = D - W) first.
    
    Args:
        shapes_info: list of shape dicts containing 'features', 'mesh', and 'pos'
        dim: feature dimension index to visualize
        top_percentile: threshold for activation highlighting (e.g. 80 = top 20%)
        apply_laplacian: whether to apply KNN Graph Laplacian smoothing
        as_mesh: if True, renders full 3D surface mesh; if False, renders point cloud
    """
    feat_list = []
    for s in shapes_info:
        f = s["features"]
        if apply_laplacian:
            f = compute_knn_laplacian_smoothing(f, s["pos"], k=15, alpha=0.3)
        feat_list.append(f[:, dim])
        
    all_v = np.concatenate(feat_list)
    g_min, g_max = all_v.min(), all_v.max()
    threshold = np.percentile(all_v, top_percentile)
    
    dim_type = "ASMAE" if dim < 128 else "DINOv2"
    mode_str = "MESH" if as_mesh else "POINT CLOUD"
    print(f"[{mode_str}] dim={dim} ({dim_type}) | Laplacian={apply_laplacian} | top {100-top_percentile}% threshold={threshold:.3f}")
    
    n = len(shapes_info)
    d_plot = None
    for i, s in enumerate(shapes_info):
        vals = feat_list[i]
        # Base color for un-activated regions: Light Gray [0.85, 0.85, 0.85]
        colors = np.full((len(vals), 3), 0.85, dtype=np.float64)
        
        mask = vals >= threshold
        if mask.any():
            norm_vals = (vals[mask] - threshold) / (g_max - threshold + 1e-8)
            colors[mask, 0] = 1.0
            colors[mask, 1] = 0.85 * (1.0 - norm_vals)
            colors[mask, 2] = 0.85 * (1.0 - norm_vals)
            
        if as_mesh:
            # Retrieve mesh vertices and triangle faces from TriMesh
            verts = s["mesh"].vertlist if hasattr(s["mesh"], "vertlist") else s["pos"]
            faces = s["mesh"].facelist if hasattr(s["mesh"], "facelist") else s["mesh"].faces
            
            # Render 3D Surface Mesh
            if d_plot is None:
                d_plot = mp.subplot(verts, faces, c=colors, s=[1, n, i])
            else:
                mp.subplot(verts, faces, c=colors, s=[1, n, i], data=d_plot)
        else:
            # Render Point Cloud (passing pos without mesh faces)
            if d_plot is None:
                d_plot = mp.subplot(s["pos"], c=colors, s=[1, n, i], shading={"point_size": 0.03})
            else:
                mp.subplot(s["pos"], c=colors, s=[1, n, i], data=d_plot, shading={"point_size": 0.03})

    prefix = "mesh" if as_mesh else "pointcloud"

# Backward-compatible alias
visualize_pointcloud_dim = visualize_feature_dim

In [7]:
# Render Mesh Feature Visualization (set as_mesh=False if you want point cloud)
visualize_feature_dim(shapes_info, dim=5, top_percentile=80, apply_laplacian=True, as_mesh=True)

[MESH] dim=5 (ASMAE) | Laplacian=True | top 20% threshold=0.046


In [8]:
# Render Mesh Feature Visualization (set as_mesh=False if you want point cloud)
visualize_feature_dim(shapes_info, dim=500, top_percentile=80, apply_laplacian=True, as_mesh=True)

[MESH] dim=500 (DINOv2) | Laplacian=True | top 20% threshold=0.028


In [9]:
# ==============================================================================
# 1-to-1 Correspondence Transfer (Supports both 3D Mesh and Point Cloud)
# ==============================================================================
def visualize_correspondence(src, tgt, as_mesh=True):
    """
    Computes and visualizes 1-to-1 correspondence transfer from source to target.
    
    Args:
        src: dict for source shape containing 'features', 'pos', 'mesh', 'name'
        tgt: dict for target shape containing 'features', 'pos', 'mesh', 'name'
        as_mesh: if True, renders full 3D surface mesh; if False, renders point cloud
        save_html: if True, saves an interactive HTML file
    """
    # 1. Extract and cast normalized features
    src_feats = src["features"].astype(np.float64)
    tgt_feats = tgt["features"].astype(np.float64)

    # 2. Compute Cosine Similarity & Nearest Neighbor P2P map (Target -> Source)
    # tgt_feats @ src_feats.T shape: [N_tgt, N_src]
    p2p = np.argmax(tgt_feats @ src_feats.T, axis=1)

    # 3. Canonical XYZ color transfer function
    def visu_xyz(pos):
        mn, mx = pos.min(axis=0), pos.max(axis=0)
        return ((pos - mn) / (mx - mn + 1e-8)).astype(np.float64)

    # 4. Color generation
    cmap1 = visu_xyz(src["pos"])
    cmap2 = np.ascontiguousarray(cmap1[p2p], dtype=np.float64)

    mode_str = "MESH" if as_mesh else "POINT CLOUD"
    print(f"[{mode_str}] 1-to-1 Correspondence: {src['name']} -> {tgt['name']}")

    # 5. Dual-mode rendering
    if as_mesh:
        # Retrieve mesh vertices and triangle faces from TriMesh
        src_verts = src["mesh"].vertlist if hasattr(src["mesh"], "vertlist") else src["pos"]
        src_faces = src["mesh"].facelist if hasattr(src["mesh"], "facelist") else src["mesh"].faces
        tgt_verts = tgt["mesh"].vertlist if hasattr(tgt["mesh"], "vertlist") else tgt["pos"]
        tgt_faces = tgt["mesh"].facelist if hasattr(tgt["mesh"], "facelist") else tgt["mesh"].faces

        # Render as 3D surface mesh with faces
        d = mp.subplot(src_verts, src_faces, c=cmap1, s=[1, 2, 0])
        mp.subplot(tgt_verts, tgt_faces, c=cmap2, s=[1, 2, 1], data=d)
    else:
        # Render as raw 3D point clouds (without faces)
        d = mp.subplot(src["pos"], c=cmap1, s=[1, 2, 0], shading={"point_size": 0.03})
        mp.subplot(tgt["pos"], c=cmap2, s=[1, 2, 1], data=d, shading={"point_size": 0.03})
    return p2p, d


In [10]:
# ==============================================================================
# Usage: Choose between Mesh or Point Cloud by changing `as_mesh`
# ==============================================================================
src = shapes_info[0]
tgt = shapes_info[1]

# Render as 3D Surface Mesh:
p2p, d = visualize_correspondence(src, tgt, as_mesh=True)

[MESH] 1-to-1 Correspondence: 9 -> 31
